In [1]:
import os, sys

# Point Spark to Java 17, Hadoop/winutils and the Python in this environment
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = "C:/Hadoop/hadoop-3.3.1"
os.environ["PATH"] += ";C:/Hadoop/hadoop-3.3.1/bin"
os.environ["PYSPARK_HADOOP_VERSION"] = "without"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from cassandra.cluster import Cluster
cluster = Cluster(["127.0.0.1"], port=9042)
# Connect to Cassandra running in Docker

session = cluster.connect()

# Create a keyspace and a small test table
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS test_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};
""")
session.execute("""
    CREATE TABLE IF NOT EXISTS test_keyspace.test_table (id int PRIMARY KEY, name text);
""")
print("Cassandra OK")



Cassandra OK


In [3]:
from pyspark.sql import SparkSession

# Start Spark with the Spark-Cassandra connector (downloaded automatically the first time)
spark = (SparkSession.builder
         .appName("CassandraTest")
         .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
         .config("spark.cassandra.connection.host", "127.0.0.1")
         .config("spark.cassandra.connection.port", "9042")
         .getOrCreate())
print("Spark OK") 

Spark OK


In [5]:
# Write two rows to Cassandra via Spark
df = spark.createDataFrame([(1, "Maryan"), (2, "Test")], ["id", "name"])
(df.write.format("org.apache.spark.sql.cassandra")
   .options(table="test_table", keyspace="test_keyspace")
   .mode("append").save())

# Read them back
(spark.read.format("org.apache.spark.sql.cassandra")
   .options(table="test_table", keyspace="test_keyspace")
   .load().show())

+---+------+
| id|  name|
+---+------+
|  1|Maryan|
|  2|  Test|
+---+------+

